In [ ]:
!pip install langchain langchain-openai pydantic

In [3]:
import os
import getpass

def _set_env(var: str):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"{var}: ")

_set_env("OPENAI_API_KEY")

In [4]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

C:\Users\ruthi\AppData\Roaming\Python\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
from pydantic import BaseModel, Field

class SearchQuery(BaseModel):
    search_query: str = Field(
        None, description="Query optimized for web search."
    )
    justification: str = Field(
        None, description="Why this query is relevant."
    )

In [6]:
structured_llm = llm.with_structured_output(SearchQuery)

In [7]:
output = structured_llm.invoke(
    "How does Calcium CT score relate to high cholesterol?"
)

output

SearchQuery(search_query='Calcium CT score and high cholesterol relationship', justification='Understanding the relationship between Calcium CT scores and high cholesterol is important for assessing cardiovascular risk. A Calcium CT score measures the amount of calcified plaque in the coronary arteries, which can be influenced by cholesterol levels. This query will help find relevant studies and articles that explain how these two factors are interconnected.')

In [8]:
def multiply(a: int, b: int) -> int:
    return a * b

In [ ]:
# Bind LLM with the tool
llm_with_tools = llm.bind_tools([multiply])

In [30]:
msg = llm_with_tools.invoke("What is 2 times 3?")
msg

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 48, 'total_tokens': 65, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_de7acce317', 'id': 'chatcmpl-DZszLLgtI1cGhh7HqmnxZvZmKkE2i', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019dd7fd-ca61-71f3-bf4a-8ba24f1731e0-0', tool_calls=[{'name': 'multiply', 'args': {'a': 2, 'b': 3}, 'id': 'call_NXD82bmEQCmjXVeTK5WKsytj', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 48, 'output_tokens': 17, 'total_tokens': 65, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [31]:
# inspect the tools calls
msg.tool_calls

[{'name': 'multiply',
  'args': {'a': 2, 'b': 3},
  'id': 'call_NXD82bmEQCmjXVeTK5WKsytj',
  'type': 'tool_call'}]

In [ ]:
tool_call = msg.tool_calls[0]

result = multiply(**tool_call["args"]) # only tool execution, no LLM call
print(result)

6


In [34]:
from langchain_core.messages import ToolMessage

tool_message = ToolMessage(
    content=str(result),                 # tool result
    tool_call_id=tool_call["id"]         # must match
)
tool_message


ToolMessage(content='6', tool_call_id='call_NXD82bmEQCmjXVeTK5WKsytj')

In [37]:
# USER -> LLM (tool call) -> TOOL -> LLM (final response)   
from langchain_core.messages import HumanMessage

user_msg = HumanMessage(content="What is 2 times 3?")

final_response = llm_with_tools.invoke([
    user_msg,    
    msg,
    tool_message
])

print(final_response.content)

2 times 3 is 6.
